# 실기 대비
# 실전 문제풀이
# set 3

## 1) 데이터 및 시나리오

### 시중 스마트폰 상세 정보

> 갓 입사한 신입사원 김모씨는 기획 부서에 배치되었다. 첫 프로젝트로 신규 스마트폰 스펙 기획 업무를 보조하게 되었다. 장고 끝에 김씨는 온라인 디지털 마켓 사이트 `디판다요`에서 판매중인 스마트폰 데이터를 수집하여 이를 분석하고 향후 프로젝트의 빅데이터로 활용하기로 결심하였다.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `mobiles.csv` | 430 | 11 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `screen_size` | string | 화면 크기 |
| `ROM` | int | 저장 공간 용량 |
| `RAM` | int | RAM 용량 |
| `num_rear_camera` | int | 후면 카메라 개수 |
| `num_front_camera` | int | 전면 카메라 개수 |
| `battery_capacity` | int | 배터리 용량 |
| `ratings` | float | 평가 점수 평균 |
| `num_of_ratings` | int | 평가 개수 |
| `sales_price` | int | 판매가격 |
| `discount_percent` | float | 할인율 |
| `sales` | float | 판매 지수 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import MinMaxScaler` |
| `from sklearn.model_selection import train_test_split` |
| `from sklearn.neighbors import KNeighborsRegressor` |
| `from sklearn.metrics import mean_squared_error` |

In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

import pandas as pd
import numpy as np

df = pd.read_csv('../../dataset/mobiles.csv')

In [3]:
#SCDI
display(df.shape)
display(df.columns)
display(df.dtypes)
display(df.isna().sum())

(430, 11)

Index(['screen_size', 'ROM', 'RAM', 'num_rear_camera', 'num_front_camera',
       'battery_capacity', 'ratings', 'num_of_ratings', 'sales_price',
       'discount_percent', 'sales'],
      dtype='object')

screen_size          object
ROM                   int64
RAM                   int64
num_rear_camera       int64
num_front_camera      int64
battery_capacity      int64
ratings             float64
num_of_ratings        int64
sales_price           int64
discount_percent    float64
sales               float64
dtype: object

screen_size         0
ROM                 0
RAM                 0
num_rear_camera     0
num_front_camera    0
battery_capacity    0
ratings             0
num_of_ratings      0
sales_price         0
discount_percent    0
sales               0
dtype: int64

### Q01.

스마트폰의 경우 많은 제품이 출시되지만 정작 주목받는 제품은 극히 적다고 한다.  
판매지수(`sales`)를 기준으로 이상치라고 판단되는 제품을 주목받는 제품이라고 판단하고 해당 제품들의 성능지표를 산출하시오.  
산출된 성능지표의 평균은 얼마인가?

#### 성능지표 계산식

$$
\text{성능지표}
=
\frac{ROM}{32}
+
\frac{RAM}{2}
+
\text{카메라 개수}
+
\frac{\text{배터리 용량}(battery\_capacity)}{1000}
$$

※ 카메라 개수: `num_rear_camera + num_front_camera`  
※ 이상치는 평균으로부터 2 표준편차보다 큰 값으로 정의한다.  
※ 결과는 반올림하여 소수점 둘째 자리까지 계산하시오. `(정답 예시: 0.12)`

In [4]:
df_q1 = df.copy()
# Z = (x - m)/std 
cond = (df_q1['sales'] - df_q1['sales'].mean())/df_q1['sales'].std() > 2.0
df_q1_cond = df_q1.loc[cond, :]
display(df_q1.shape, df_q1_cond.shape)

#덧셈을 함부로 줄바꿈하면 안된다.
ser = df_q1_cond['ROM']/32 + df_q1_cond['RAM']/2 + (df_q1_cond['num_rear_camera'] + df_q1_cond['num_front_camera']) + df_q1_cond['battery_capacity']/1000
display(type(ser))
round(ser.mean(), 2)

(430, 11)

(16, 11)

pandas.core.series.Series

11.01

### Q02.

판매 지수(`sales`)와 가장 상관관계가 높은 변수를 찾고자 한다.  
배터리 용량(`battery_capacity`), 평가 점수 평균(`ratings`), 평가 개수(`num_of_ratings`), 판매 가격(`sales_price`), 할인율(`discount_percent`) 변수와 판매 지수(`sales`)를 피어슨 상관분석을 실시하였을 때 상관계수의 절대값이 가장 큰 변수의 상관계수는 얼마인가?

※ 후면 카메라가 1개인 제품은 제외하시오.  
※ 통계적 유의성은 고려하지 않음.  
※ 결과는 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [5]:
df_q2 = df.copy()
cond = ~(df_q2['num_rear_camera'] == 1)
df_q2_1 = df_q2.loc[cond,:]
display(df_q2.shape, df_q2_1.shape)
df_q2_corr = df_q2_1.corr(method='pearson')['sales'].abs().drop(index='sales')
display(df_q2_corr)
round(df_q2_corr.max(),2)
# 상관관계에서 col을 분류하고 상관관계할 필요 없다. 어차피 sales 열에 대한 게 각각 계산되기때문
#cols = ['battery_capacity', 'ratings', 'num_of_ratings', 'sales_price','discount_percent', 'sales']
#df_q2_2 = df_q2_1[cols].copy()
#df_q2_2_corr = df_q2_2.corr(method='pearson')['sales'].abs()
#display(df_q2_2_corr)

(430, 11)

(390, 11)

ROM                 0.223810
RAM                 0.188759
num_rear_camera     0.113677
num_front_camera    0.108239
battery_capacity    0.025680
ratings             0.226075
num_of_ratings      0.949114
sales_price         0.247760
discount_percent    0.223471
Name: sales, dtype: float64

0.95

### Q03.

판매 지수(`sales`)를 예측하기 위해 머신러닝 모델을 활용하고자 한다.  
k-NN 알고리즘을 사용하고 연산에 사용하는 이웃의 개수를 변화하면서 가장 성능이 좋은 모델을 확보하려 한다.  
RMSE(Root Mean Squared Error)를 기준으로 가장 성능이 좋은 모델을 확인하고 해당 모델의 k(이웃 개수)를 구하라.

#### 독립변수 / 종속변수

| 구분 | 변수 |
|---|---|
| 독립변수 | 판매 지수를 제외한 모든 변수 |
| 종속변수 | 판매 지수 |

※ 명목형 독립변수의 경우 One Hot Encoding을 실시한 결과를 모델에 사용하시오.  
※ 학습에 사용하는 독립변수의 개수는 총 14개이다.  
※ 학습 및 평가 데이터 세트 분할비는 8:2로 하시오.  
※ 정규화는 Min-Max 정규화를 실시하며 평가 데이터 세트는 학습 데이터 세트 기반으로 정규화 하시오.  
※ seed는 `123`으로 고정하시오.  
※ 최근접 이웃은 3, 5, 7, 9, 11개를 사용하시오.  
※ 정답은 자연수로 출력하시오. `(정답 예시: 7)`

In [27]:
df_q3 = df.copy()
X = df_q3.drop(columns =['sales']).copy()
y = df_q3['sales'].copy()

X_dummies = pd.get_dummies(X, columns=['screen_size'])
display(X.shape,X['screen_size'].nunique(), X_dummies.shape)
display(X_dummies.columns)
X_dummies.columns = X_dummies.columns.str.replace(" ","_")
display(X_dummies)
#특정 컬럼이름 바꾸기
#df.rename(columns={"ROM" : "RRRR"})

#D
X_train, X_test, y_train, y_test = train_test_split(X_dummies,y, train_size=0.8, random_state = 123)
display(X_train.dtypes)

#N
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_n = scaler.transform(X_train)
X_test_n = scaler.transform(X_test)
#M
dict_rmse ={}
for k in [3,5,7,9,11] :
    model = KNeighborsRegressor(n_neighbors = k)
    model.fit(X_train_n, y_train)
    y_pred = model.predict(X_test_n)
    dict_rmse[k] = mean_squared_error(y_test, y_pred)**0.5
ser = pd.Series(dict_rmse, name = 'rmse')
display(ser)
ser = ser.sort_values() #dataframe 공통
display(ser.idxmin())
#E

(430, 10)

5

(430, 14)

Index(['ROM', 'RAM', 'num_rear_camera', 'num_front_camera', 'battery_capacity',
       'ratings', 'num_of_ratings', 'sales_price', 'discount_percent',
       'screen_size_Large', 'screen_size_Medium', 'screen_size_Small',
       'screen_size_Very Large', 'screen_size_Very Small'],
      dtype='object')

,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,screen_size_Large,screen_size_Medium,screen_size_Small,screen_size_Very_Large,screen_size_Very_Small
0,64,2,1,1,1800,4.5,38645,32999,0.17,0,0,0,0,1
1,64,4,2,1,2815,4.5,244,57149,0.04,0,0,1,0,0
2,64,2,1,1,1800,4.5,38645,32999,0.17,0,0,0,0,1
3,64,3,1,1,2942,4.6,5366,42999,0.10,0,1,0,0,0
4,128,4,2,1,2815,4.6,745,69149,0.02,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,32,3,2,1,4000,4.3,1870,7999,0.30,0,0,1,0,0
426,64,4,2,1,4000,4.3,1783,9699,0.28,0,0,1,0,0
427,128,6,3,1,4250,4.2,1554,21999,0.12,1,0,0,0,0
428,32,3,2,1,5000,4.2,8161,8299,0.07,0,1,0,0,0


ROM                         int64
RAM                         int64
num_rear_camera             int64
num_front_camera            int64
battery_capacity            int64
ratings                   float64
num_of_ratings              int64
sales_price                 int64
discount_percent          float64
screen_size_Large           uint8
screen_size_Medium          uint8
screen_size_Small           uint8
screen_size_Very_Large      uint8
screen_size_Very_Small      uint8
dtype: object

3     40.440549
5     48.800827
7     53.186755
9     55.484384
11    56.160703
Name: rmse, dtype: float64

3